In [11]:
# === STEP 1: Import library & konfigurasi dasar ===
import requests
import time
import pandas as pd

# URL utama API Qontak (versi 3.1)
API_BASE = "https://app.qontak.com/api/v3.1"
TOKEN_URL = "https://app.qontak.com/oauth/token"

# Masukkan kredensial API kamu di sini
CLIENT_ID = "873f6ef62e89c9fc90b7cea0cb0871ed39a9f290f08dc0528038aba26d8eb423"
CLIENT_SECRET = "4a333804b561a61966ff5eb6d1e1cdba6db7ebb7fa0098fecac2101cca0d8ec1"
USERNAME = "sitinurul.cias@gmail.com"
PASSWORD = "Nurul123!@#"

# Tempat menyimpan token & masa berlaku
token_info = {
    "access_token": None,
    "refresh_token": None,
    "expires_at": 0  # waktu (epoch) kapan token kadaluarsa
}


In [12]:
# === STEP 2: Fungsi Autentikasi dan Refresh Token ===

def authenticate():
    """Login awal untuk mendapatkan access_token dan refresh_token."""
    payload = {
        "grant_type": "password",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "username": USERNAME,
        "password": PASSWORD
    }
    response = requests.post(TOKEN_URL, data=payload)
    if response.status_code != 200:
        raise Exception(f"Gagal autentikasi: {response.status_code} {response.text}")

    result = response.json()
    token_info["access_token"] = result.get("access_token")
    token_info["refresh_token"] = result.get("refresh_token")
    token_info["expires_at"] = time.time() + result.get("expires_in", 3600) - 60
    print("✅ Token berhasil dibuat!")


def refresh_access_token():
    """Perbarui access_token menggunakan refresh_token."""
    if not token_info["refresh_token"]:
        raise Exception("Tidak ada refresh_token, lakukan autentikasi ulang.")
    
    payload = {
        "grant_type": "refresh_token",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "refresh_token": token_info["refresh_token"]
    }
    response = requests.post(TOKEN_URL, data=payload)
    if response.status_code != 200:
        raise Exception(f"Gagal refresh token: {response.status_code} {response.text}")

    result = response.json()
    token_info["access_token"] = result.get("access_token")
    token_info["refresh_token"] = result.get("refresh_token", token_info["refresh_token"])
    token_info["expires_at"] = time.time() + result.get("expires_in", 3600) - 60
    print("♻️ Token berhasil diperbarui!")


def get_access_token():
    """Pastikan token masih valid, jika tidak maka refresh dulu."""
    if not token_info["access_token"]:
        authenticate()
    elif time.time() >= token_info["expires_at"]:
        refresh_access_token()
    return token_info["access_token"]


In [13]:
# === STEP 3: Fungsi untuk ambil data tasks ===

def get_tasks(filter_value="alltask", page=1, per_page=25):
    """Mengambil daftar task dari API Qontak."""
    token = get_access_token()
    url = f"{API_BASE}/tasks"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }
    params = {
        "filter": filter_value,
        "page": page,
        "per_page": per_page
    }

    response = requests.get(url, headers=headers, params=params)

    # Jika token invalid atau expired
    if response.status_code == 401:
        refresh_access_token()
        token = token_info["access_token"]
        headers["Authorization"] = f"Bearer {token}"
        response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        raise Exception(f"Gagal ambil tasks: {response.status_code} {response.text}")

    data = response.json()
    print("✅ Data task berhasil diambil!")
    return data


# === Uji fungsi ===
tasks_data = get_tasks()

# Tampilkan hasil
df = pd.DataFrame(tasks_data.get("data", []))
display(df.head())


Exception: Gagal autentikasi: 401 {"error":"invalid_client","error_description":"Client authentication failed due to unknown client, no client authentication included, or unsupported authentication method."}

In [14]:
import requests
import pandas as pd

# === Ganti dengan API Token dari Qontak CRM Dashboard ===
API_TOKEN = "873f6ef62e89c9fc90b7cea0cb0871ed39a9f290f08dc0528038aba26d8eb423"

# === URL Endpoint CRM (bukan app.qontak) ===
url = "https://crm.qontak.com/api/v3.1/tasks"
params = {
    "filter": "alltask",
    "page": 1,
    "per_page": 25
}
headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Accept": "application/json"
}

# === Ambil data task ===
response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    print("✅ Data task berhasil diambil!")
    
    # ubah ke DataFrame agar mudah dilihat
    df = pd.DataFrame(data.get("data", []))
    display(df.head())
else:
    print("❌ Gagal ambil data task:", response.status_code, response.text)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [17]:
import requests

API_TOKEN = "873f6ef62e89c9fc90b7cea0cb0871ed39a9f290f08dc0528038aba26d8eb423"

url = "https://crm.qontak.com/api/v3.1/tasks"
headers = {"Authorization": f"Bearer {API_TOKEN}"}
params = {"filter": "alltask", "page": 1, "per_page": 25}

response = requests.get(url, headers=headers, params=params)

# Cek dulu status dan tipe response
print("Status code:", response.status_code)
print("Response content-type:", response.headers.get("Content-Type"))
print("Response text (preview):", response.text[:300])  # lihat sebagian isi

if response.status_code == 200 and "application/json" in response.headers.get("Content-Type", ""):
    data = response.json()
    print("✅ Data berhasil diambil:", data)
else:
    print("❌ Respons bukan JSON, mungkin token salah atau expired.")


Status code: 200
Response content-type: text/html
Response text (preview): <!doctype html>
<html lang="en" data-n-head="%7B%22lang%22:%7B%221%22:%22en%22%7D%7D">
  <head>
    <meta data-n-head="1" data-hid="robots" name="robots" content="noindex, nofollow"><meta data-n-head="1" charset="utf-8"><meta data-n-head="1" name="viewport" content="width=device-width,initial-scale=
❌ Respons bukan JSON, mungkin token salah atau expired.


In [4]:
import base64
import requests
import pandas as pd

# === STEP 1. MASUKKAN KREDENSIAL API ===
CLIENT_ID = "873f6ef62e89c9fc90b7cea0cb0871ed39a9f290f08dc0528038aba26d8eb423"
CLIENT_SECRET = "4a333804b561a61966ff5eb6d1e1cdba6db7ebb7fa0098fecac2101cca0d8ec1"

# === STEP 2. ENCODE CLIENT ID & SECRET KE BASE64 (untuk Basic Auth) ===
auth_string = f"{CLIENT_ID}:{CLIENT_SECRET}"
auth_bytes = base64.b64encode(auth_string.encode("ascii")).decode("ascii")

# === STEP 3. DAPATKAN ACCESS TOKEN ===
token_url = "https://app.qontak.com/oauth/token"
headers = {
    "Authorization": f"Basic {auth_bytes}",
    "Content-Type": "application/x-www-form-urlencoded"
}
data = {"grant_type": "client_credentials"}

response = requests.post(token_url, headers=headers, data=data)

if response.status_code == 200:
    token_data = response.json()
    access_token = token_data["access_token"]
    print("✅ Access token berhasil diambil!")
else:
    print("❌ Gagal autentikasi:", response.status_code, response.text)
    raise SystemExit

# === STEP 4. GUNAKAN TOKEN UNTUK AMBIL DATA TASKS ===
tasks_url = "https://app.qontak.com/api/v3.1/tasks"
params = {"filter": "alltask", "page": 1, "per_page": 25}
headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json"
}

tasks_resp = requests.get(tasks_url, headers=headers, params=params)

if tasks_resp.status_code == 200:
    tasks = tasks_resp.json()
    print("✅ Data task berhasil diambil!")

    # === STEP 5. OPSIONAL: SIMPAN KE DATAFRAME ===
    df = pd.DataFrame(tasks.get("data", []))
    print("\n📋 Daftar Task (5 pertama):")
    print(df.head())
else:
    print("❌ Gagal ambil data task:", tasks_resp.status_code, tasks_resp.text)


❌ Gagal autentikasi: 401 {"error":"invalid_client","error_description":"Client authentication failed due to unknown client, no client authentication included, or unsupported authentication method."}


SystemExit: 

c:\Users\IP SLIM3\anaconda3\envs\data_assistant\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [18]:
import requests
import pandas as pd

# === Masukkan token CRM yang BARU dari dashboard ===
API_TOKEN = "873f6ef62e89c9fc90b7cea0cb0871ed39a9f290f08dc0528038aba26d8eb423"

url = "https://crm.qontak.com/api/v3.1/tasks"
params = {"filter": "alltask", "page": 1, "per_page": 25}
headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers, params=params)

# Deteksi apakah response valid JSON atau HTML login
if response.headers.get("Content-Type", "").startswith("application/json"):
    data = response.json()
    print("✅ Data task berhasil diambil!")
    df = pd.DataFrame(data.get("data", []))
    display(df.head())
else:
    print("❌ Respons bukan JSON. Kemungkinan besar token salah atau sudah expired.")
    print("Status:", response.status_code)
    print("Preview:", response.text[:300])


❌ Respons bukan JSON. Kemungkinan besar token salah atau sudah expired.
Status: 200
Preview: <!doctype html>
<html lang="en" data-n-head="%7B%22lang%22:%7B%221%22:%22en%22%7D%7D">
  <head>
    <meta data-n-head="1" data-hid="robots" name="robots" content="noindex, nofollow"><meta data-n-head="1" charset="utf-8"><meta data-n-head="1" name="viewport" content="width=device-width,initial-scale=
